In [ ]:
import datetime
from argparse import ArgumentParser
from pathlib import Path
import h5py
import numpy as np
import pandas as pd
from pynwb import NWBHDF5IO
from sklearn.model_selection import train_test_split


from torch_brain.data import (
    BrainsetDescription,
    Data,
    DeviceDescription,
    Interval,
    RegularTimeSeries,
    SessionDescription,
    serialize_fn_map,
)
from torch_brain.pipeline import BrainsetPipeline
from torch_brain.utils.dandi import (
    download_file,
    extract_spikes_from_nwbfile,
    extract_subject_from_nwb,
    get_nwb_asset_list,
)

brainset_id = "nlb_area2_2021"
fpath = r"C:\Workspace\SSM-BCI-Repo\data\raw\nlb_area2_2021\sub-Han_desc-train_behavior+ecephys.nwb"
processed_dir = Path(r"C:\Workspace\SSM-BCI-Repo\data\processed\nlb_area2_2021")

ModuleNotFoundError: No module named 'sklearn'

In [93]:
import numpy as np

def split_trials(trials, test_size=0.2, valid_size=0.1):
    num_trials = len(trials)
    
    # Calculate the exact splitting index boundaries
    train_end = int(num_trials * (1.0 - test_size - valid_size))
    valid_end = int(num_trials * (1.0 - test_size))
    
    # Create the sequential index arrays
    train_ids = np.arange(0, train_end)
    valid_ids = np.arange(train_end, valid_end)
    test_ids = np.arange(valid_end, num_trials)
    
    # Select the trials using your mask method
    train_trials = trials.select_by_mask(np.isin(np.arange(num_trials), train_ids))
    valid_trials = trials.select_by_mask(np.isin(np.arange(num_trials), valid_ids))
    test_trials = trials.select_by_mask(np.isin(np.arange(num_trials), test_ids))
    
    return train_trials, valid_trials, test_trials

def extract_behavior(nwbfile):
    """Extract behavior from the NWB file.

    ..note::
        - Only care about hand velocity
        - Timestamps not present in the NWB file
    """
    # cursor, hand and eye share the same timestamps (verified)
    raw_hand_vel = nwbfile.processing["behavior"]["hand_vel"].data[:]

    # These samples are uniformly sampled at 1000Hz,
    samp_rate = 1e3
    num_timesteps = raw_hand_vel.shape[0]
    start_time = 0.0

    hand = RegularTimeSeries(
        sampling_rate=samp_rate,
        # pos=hand_pos,
        vel=raw_hand_vel,
        domain="auto",
        domain_start=start_time,
    )
    
    return hand



def extract_center_out_reaching_trials(nwbfile, hand):
    r"""Extract trial information from the NWB file. Trials that are flagged as
    "to discard" or where the monkey failed are marked as invalid."""
    trial_table = nwbfile.trials.to_dataframe()

    # infer the trial structure from the trial table
    trial_grid = np.append(
        trial_table.target_on_time.iloc[:],
        min(trial_table.stop_time.iloc[-1] + 1.0, hand.domain.end[-1]),
    )

    default_value = np.append(
        trial_table.start_time.iloc[0], trial_table.stop_time.values + 1.0
    )
    default_value[1:-1] = np.minimum(
        default_value[1:-1], trial_table.stop_time.values[1:]
    )
    nan_mask = np.isnan(trial_grid)
    trial_grid[nan_mask] = default_value[nan_mask]
    trial_grid = trial_grid.astype(np.float64)
    trial_table["end"] = trial_grid[1:]
    trial_table["start"] = trial_grid[:-1]

    trial_table = trial_table.rename(
        columns={
            "split": "split_indicator",
        }
    )

    trials = Interval.from_dataframe(trial_table)
    assert trials.is_disjoint()

    # find valid trials
    success_mask = trials.result == "R"
    # valid_target_mask = ~np.isnan(trials.target_id)
    max_duration_mask = (trials.end - trials.start) < 6.0
    min_duration_mask = (trials.end - trials.start) > 0.5

    trials.is_valid = (
        success_mask & max_duration_mask & min_duration_mask
        # success_mask & valid_target_mask & max_duration_mask & min_duration_mask
    )

    valid_trials = trials.select_by_mask(trials.is_valid)

    # isolate movement phases
    movement_phases = Data(
        hold_period=Interval(
            start=valid_trials.target_on_time, end=valid_trials.go_cue_time
        ),
        reach_period=Interval(
            start=valid_trials.go_cue_time, end=valid_trials.stop_time
        ),
        return_period=Interval(start=valid_trials.stop_time, end=valid_trials.end),
        invalid=trials.select_by_mask(~trials.is_valid),
        domain="auto",
    )

    # everything outside of the different identified periods will be marked as random
    movement_phases.random_period = hand.domain.difference(movement_phases.domain)

    return trials, movement_phases


brainset_description = BrainsetDescription(
    id=brainset_id,
    origin_version="dandi/000127/0.220113.0359",
    derived_version="2.0.0",
    source="https://dandiarchive.org/dandiset/000127",
    description="This dataset contains sorted unit spiking times and behavioral data from " \
    "a macaque performing a reaching task with perturbations. In the experimental task, the " \
    "subject performed delayed center-out reaches using a manipulandum to control a cursor. " \
    "On a portion of the trials, the manipulandum applied a bump during the center hold prior " \
    "to the reach. Neural activity was recorded from an electrode array implanted in somatosensory " \
    "area 2. Hand position, cursor position, force applied to the manipulandum, length and velocity " \
    "of various arm muscles, and angle and velocity of various arm joints were all recorded during " \
    "the experiment. Provided as part of the Neural Latents Benchmark: https://neurallatents.github.io.",
)

# open file
io = NWBHDF5IO(fpath, "r")
nwbfile = io.read()
subject = extract_subject_from_nwb(nwbfile)

recording_date = nwbfile.session_start_time.strftime("%Y%m%d")
device_id = f"{subject.id}_{recording_date}"
session_id = f"{subject.id}_area2"
if "test" in str(fpath):
    session_id += "_test"
else:
    session_id += "_train"

store_path = processed_dir / f"{session_id}.h5"
if store_path.exists():
    assert f"{store_path} already exists. Do you want to overwrite it? (y/n)"

# register session
session_description = SessionDescription(
    id=session_id,
    recording_date=datetime.datetime.strptime(recording_date, "%Y%m%d"),
    task="REACHING",
)

# register device
device_description = DeviceDescription(
    id=device_id,
    recording_tech="UTAH_ARRAY_SPIKES",
)

# extract spiking activity
# this data is from dandi, we can use our helper function
spikes, units = extract_spikes_from_nwbfile(
    nwbfile,
    recording_tech="UTAH_ARRAY_SPIKES",
)

# extract behaviour
hand = extract_behavior(nwbfile)

# extract data about trial structure
# trials = extract_trials(nwbfile)
trials, movement_phases = extract_center_out_reaching_trials(
    nwbfile, 
    hand
)

data = Data(
    brainset=brainset_description,
    session=session_description,
    device=device_description,
    # neural activity
    spikes=spikes,
    units=units,
    # stimuli and behavior
    trials=trials,
    movement_phases=movement_phases,
    hand=hand,
    # domain
    domain=hand.domain,
)

if "test" not in str(fpath):
    print("-"*50)
    print("Creating splits...")
    _, valid_trials, test_trials = split_trials(
        trials.select_by_mask(trials.is_valid),
        test_size=0.2,
        valid_size=0.1
    )
    train_sampling_intervals = data.domain.difference(
        (valid_trials | test_trials).dilate(1.0)
    )

    data.train_domain = train_sampling_intervals
    data.valid_domain = valid_trials
    data.test_domain = test_trials

    io.close()

    print(train_sampling_intervals.start)
    print(train_sampling_intervals.end)
    print("-"*50)
    print(valid_trials.start)
    print(valid_trials.end)
    print("-"*50)
    print(test_trials.start)
    print(test_trials.end)

# # save data to disk
# with h5py.File(store_path, "w") as file:
#     data.to_hdf5(file, serialize_fn_map=serialize_fn_map)




--------------------------------------------------
Creating splits...
[   0.    1569.081 1574.503 1592.261 1598.479 1610.082 1631.39  1650.629
 1658.801 1680.478 1695.945 1707.412 1730.599 1744.223 1783.739 1797.356
 1812.774 1822.845 1830.461 1836.745 1855.629 1869.131 1879.422 1887.28
 1891.661 1917.75  1934.745 1943.623 1963.629 1972.385 1986.82  2014.493
 2024.636 2031.343 2037.453 2044.368 2063.923 2075.049 2083.881 2094.511
 2102.621 2119.779 2125.623 2130.227 2134.132 2153.442 2161.483 2192.472
 2198.252 2202.93  2220.495]
[1565.349 1570.937 1588.554 1594.799 1599.836 1610.876 1631.517 1650.847
 1662.947 1692.264 1703.411 1712.63  1738.49  1772.399 1791.289 1808.007
 1816.164 1823.778 1832.269 1846.81  1858.001 1873.118 1883.668 1888.087
 1892.748 1922.752 1937.786 1949.596 1965.014 1983.217 1996.024 2015.849
 2027.759 2033.841 2037.894 2049.826 2067.706 2075.146 2087.451 2098.677
 2108.42  2121.951 2126.544 2130.411 2147.871 2157.73  2176.832 2194.382
 2199.177 2205.191 2221.30

C:\Users\Hsien Rong\AppData\Local\Temp\ipykernel_5276\3308950798.py:37: DeprecationWarning: The `domain` argument of `RegularTimeSeries` is deprecated and will be removed in a future version. The domain is always computed automatically as [domain_start, domain_start + len(self) / sampling_rate); you can drop `domain="auto"` from your call.
  hand = RegularTimeSeries(


In [27]:
spikes

IrregularTimeSeries(
  timestamps=[1017465],
  unit_index=[1017465]
)

In [51]:
# io = NWBHDF5IO(rf"C:\Workspace\SSM-BCI-Repo\data\raw\nlb_area2_2021\sub-Han_desc-test_ecephys.nwb", "r")
io = NWBHDF5IO(rf"C:\Workspace\SSM-BCI-Repo\data\raw\nlb_area2_2021\sub-Han_desc-train_behavior+ecephys.nwb", "r")

nwbfile = io.read()
nwbfile

Data type,object
Shape,"(4,)"
Array size,32.00 bytes
Chunk shape,None
Compression,None
Compression opts,None
Uncompressed size (bytes),32
Compressed size (bytes),64
Compression ratio,0.5
Data type,float64
Shape,"(2223449, 6)"


In [52]:
print(nwbfile.trials.to_dataframe().to_string())

     start_time  stop_time result  ctr_hold  ctr_hold_bump  bump_dir  target_on_time  target_dir  go_cue_time  bump_time  move_onset_time  cond_dir  split
id                                                                                                                                                        
0         0.502      4.556      R  1.106973          False       NaN           3.734        45.0        3.735        NaN            4.075      45.0    val
1         5.059      5.588      A  1.344188           True       NaN             NaN       225.0          NaN        NaN              NaN       NaN   none
2         6.090      9.374      I  1.170550           True     225.0           8.369       180.0        8.370      7.905            7.898     225.0   none
3         9.877     15.298      R  1.222202          False       NaN          14.318        45.0       14.319        NaN           14.656      45.0  train
4        15.800     17.005      A  0.777271          False       NaN  

In [46]:
for i in nwbfile.trials.to_dataframe()['split']:
    print(i)

test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test
test


In [85]:
trial_table = nwbfile.trials.to_dataframe()
print(trial_table.to_string())

     start_time  stop_time result  ctr_hold  ctr_hold_bump  bump_dir  target_on_time  target_dir  go_cue_time  bump_time  move_onset_time  cond_dir  split
id                                                                                                                                                        
0         0.502      4.556      R  1.106973          False       NaN           3.734        45.0        3.735        NaN            4.075      45.0    val
1         5.059      5.588      A  1.344188           True       NaN             NaN       225.0          NaN        NaN              NaN       NaN   none
2         6.090      9.374      I  1.170550           True     225.0           8.369       180.0        8.370      7.905            7.898     225.0   none
3         9.877     15.298      R  1.222202          False       NaN          14.318        45.0       14.319        NaN           14.656      45.0  train
4        15.800     17.005      A  0.777271          False       NaN  